# Notebook 06 — API Testing, Error Handling, Monitoring, and Performance

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project root:', project_root)


## Testing and Production Engineering

### Definition
Testing and Production Engineering in this project means we design, validate, serve, and observe ML predictions through stable HTTP interfaces.

### Theory
APIs decouple producers (model service) and consumers (web app, batch jobs, partner systems). Typed contracts reduce ambiguity and production failures.

### Motivation
Model quality without reliable API behavior is not production-ready.

### Real-World Example
ML fraud APIs require strict SLAs, predictable errors, and auditable request tracking.

### Visual Explanation
Diagram/code cell below shows component flow and responsibilities.

### Code Explanation
Code cells in this section are structured as: setup → implementation → validation output.

### Best Practices
- Keep contracts explicit and versioned.
- Validate early at boundary.
- Log request IDs for traceability.
- Measure latency, error rate, and throughput.

### Common Mistakes
- Manual testing only.
- Missing negative tests for malformed payloads and auth failures.


In [ ]:
import httpx

payload = {
    'MedInc': 8.3252, 'HouseAge': 41.0, 'AveRooms': 6.9841,
    'AveBedrms': 1.0238, 'Population': 322.0, 'AveOccup': 2.5556,
    'Latitude': 37.88, 'Longitude': -122.23
}

try:
    r = httpx.get('http://127.0.0.1:8000/health', timeout=3.0)
    print('Health status:', r.status_code, r.json())
    rp = httpx.post('http://127.0.0.1:8000/predict', json=payload, timeout=5.0)
    print('Predict status:', rp.status_code)
    print('Predict body keys:', list(rp.json().keys()))
except Exception as exc:
    print('API not running locally; skip live call test:', exc)

In [ ]:
import pandas as pd
from pathlib import Path

perf_path = Path('artifacts/performance/api_performance_summary.json')
if perf_path.exists():
    display(pd.read_json(perf_path, typ='series'))
else:
    print('Run scripts/benchmark_api.py after API is up to generate performance report.')

## Security Basics

- Strict Pydantic validation at boundary.
- Optional API key for protected inference endpoints.
- Rate limiting to reduce abuse burst risk.
- Request ID correlation for incident debugging.